# Resolución del 8-puzzle mediante búsqueda no informada

De un tablero desordenado al ordenado, sin conocer el grafo.

Russell & Norvig, *Artificial Intelligence: A Modern Approach*,
4ª edición, capítulo 3, sección 3.2.1 y figura 3.4.


***
## Parte 1 — Configuración del problema


### 1.1 — El problema

Un tablero de 3×3 con ocho fichas numeradas y un hueco. Se corre una ficha por
vez al hueco, hasta dejar el tablero ordenado.

Los elementos a definir:

- **INITIAL**: el tablero desordenado
- **GOAL**: el tablero ordenado
- **ACTIONS(s)**: a dónde se puede mover el hueco
- **RESULT(s, a)**: el tablero que queda después de moverlo
- **ACTION_COST(s, a, s')**: 1 siempre, mover una ficha es mover una ficha

Un **estado** es una tupla de nueve números, con el `0` como hueco:

```
(1, 2, 3,        1 2 3
 4, 8, 5,   ->   4 8 5
 7, 6, 0)        7 6 _
```

Tupla y no lista porque tiene que poder entrar en un conjunto y en un
diccionario: los algoritmos guardan estados en `reached`, y para eso el estado
tiene que ser hashable.

In [1]:
#| include: false

import sys

# Los modulos de la clase viven en la carpeta 0-lib/ de al lado.
sys.path.append("../0-lib")


In [2]:
# La interfaz abstracta del capitulo 3: INITIAL, GOAL, ACTIONS, RESULT.
from problem import Problem

# El hueco es el 0. Las posiciones van de 0 a 8, por filas.
GOAL = (1, 2, 3,
        4, 5, 6,
        7, 8, 0)

# Cuanto se mueve el hueco dentro de la tupla con cada accion.
PASOS = {"abajo": +3, "arriba": -3, "derecha": +1, "izquierda": -1}


class EightPuzzle(Problem):
    """El 8-puzzle. Una accion mueve EL HUECO, no la ficha.

    Es la convencion del libro y simplifica todo: hay a lo sumo cuatro
    acciones, y son las mismas cuatro siempre. Mover el hueco "arriba"
    se ve en el tablero como una ficha que baja.
    """

    def __init__(self, INITIAL, GOAL=GOAL):
        super().__init__(INITIAL, GOAL)

    def ACTIONS(self, state):
        i = state.index(0)     # en que casilla esta el hueco, de 0 a 8
        fila = i // 3
        col = i % 3
        # El hueco no puede salirse del tablero: contra un borde, esa
        # accion no esta. El orden es siempre el mismo, asi dos corridas
        # arman el mismo arbol.
        acciones = []
        if fila < 2:
            acciones.append("abajo")
        if fila > 0:
            acciones.append("arriba")
        if col < 2:
            acciones.append("derecha")
        if col > 0:
            acciones.append("izquierda")
        return acciones

    def RESULT(self, state, action):
        # Mover el hueco es intercambiarlo con la ficha de al lado.
        i = state.index(0)
        j = i + PASOS[action]
        fichas = list(state)
        fichas[i], fichas[j] = fichas[j], fichas[i]
        return tuple(fichas)

    # ACTION_COST no se define: cada movimiento cuesta 1, que es lo que
    # ya devuelve Problem.


Una acción mueve **el hueco**, no la ficha. Es la convención del libro y
simplifica el problema: hay a lo sumo cuatro acciones y son siempre las mismas
cuatro, en vez de ocho fichas cada una con sus movimientos posibles.

Sobre la tupla, mover el hueco es intercambiarlo con la casilla que está a un
salto de distancia:

```
arriba -3      abajo +3      izquierda -1      derecha +1
```

`ACTIONS` devuelve solo las que no se caen del tablero: con el hueco en un borde
son tres, en una esquina dos. Ojo con el nombre al mirar el tablero: mover el
hueco `arriba` se ve como una ficha que baja.

### 1.2 — Los dos tableros

- **CHICO**: a cuatro movimientos del objetivo. Los árboles de búsqueda entran en
  la pantalla, así que se puede mirar paso a paso.
- **FIGURA_34**: el de la figura 3.4 del libro, a veintiún movimientos. Ahí no
  hay nada para mirar, pero es donde los números se separan.

In [3]:
CHICO = (1, 2, 3,
         4, 8, 5,
         7, 6, 0)

# El de la figura 3.4 del libro. Veintiun movimientos: sirve para la
# tabla de numeros, no para mirar paso a paso.
FIGURA_34 = (7, 2, 4,
             0, 5, 6,
             8, 3, 1)

In [4]:
from puzzle import lado_a_lado, tablero

print(lado_a_lado(CHICO, GOAL))

1 2 3   ->   1 2 3
4 8 5   ->   4 5 6
7 6 _   ->   7 8 _


### 1.3 — La interfaz, sobre el tablero chico

El hueco arranca abajo a la derecha, que es justo su lugar en el objetivo. Para
resolver el tablero **tiene que salir de ahí y volver**: no se puede ir derecho,
y por eso la solución son cuatro movimientos y no dos.

In [5]:
problem = EightPuzzle(CHICO)

print("INITIAL")
print(tablero(problem.INITIAL))
print()
print("ACTIONS  ", problem.ACTIONS(CHICO))
print()
print("RESULT(INITIAL, 'izquierda')")
print(tablero(problem.RESULT(CHICO, "izquierda")))
print()
print("IS_GOAL(INITIAL) ", problem.IS_GOAL(CHICO))

INITIAL
1 2 3
4 8 5
7 6 _

ACTIONS   ['arriba', 'izquierda']

RESULT(INITIAL, 'izquierda')
1 2 3
4 8 5
7 _ 6

IS_GOAL(INITIAL)  False


Tener en cuenta lo que **no** hay: ningún diccionario de vecinos. En Rumania las
23 rutas están escritas una por una y `ACTIONS` las lee del mapa. Acá el grafo no
existe en ningún lado: `ACTIONS` calcula los vecinos en el momento, cada vez que
se lo llama, con una cuenta sobre la tupla.

Los tableros posibles son las 9! = 362.880 permutaciones de nueve casillas, y
desde uno dado se llega a la mitad: **181.440 estados**. Ninguno de ellos está
escrito en ninguna parte, y nunca están todos juntos en memoria.

::: {.content-hidden}
### 1.4 — Código auxiliar
Código para la visualización del problema
:::


In [6]:
#| include: false

from html import escape

from IPython.display import HTML

from web_puzzle import WebPuzzle


def ver(view, alto=820):
    """Mete la pagina que grabo el view adentro de esta celda."""
    return HTML(f'<div><iframe srcdoc="{escape(view.html(), quote=True)}" '
                f'style="width:100%; height:{alto}px; border:1px solid #dee2e6; '
                f'border-radius:.25rem" title="la busqueda paso a paso"></iframe></div>')

***
## Parte 2 — Algoritmos de búsqueda no informada


Cada página se recorre paso a paso y tiene dos vistas, con el switch de arriba a
la derecha:

- **tablero**: el estado de ahora y los estados a los que puede ir. Es el
  equivalente del mapa de Rumania, lo que se ve sin saber nada de la historia de
  la búsqueda. La ficha resaltada es la que se movió, así que la acción se lee
  sola.
- **árbol**: todo lo que la búsqueda construyó hasta ese paso. Ahí se ve la
  diferencia entre un algoritmo y otro, que en el tablero no se ve.

### 2.1 — Búsqueda en amplitud

La frontera es una cola **FIFO**: se toman primero los nodos más antiguos, así
que el árbol se recorre por niveles. Se evalúa si se llegó al objetivo **al
generar** los hijos, y se corta ahí.

Acá todos los movimientos cuestan 1, que es justo la condición para que eso sea
óptimo: cuando aparece el objetivo, los niveles anteriores ya se recorrieron
enteros, así que no puede haber una solución con menos movimientos.

In [7]:
from breadth_first_search import breadth_first_search
from node import path_states

solucion_amplitud = breadth_first_search(problem)

print(lado_a_lado(*path_states(solucion_amplitud)))
print("costo:", solucion_amplitud.PATH_COST)


1 2 3   ->   1 2 3   ->   1 2 3   ->   1 2 3   ->   1 2 3
4 8 5   ->   4 8 5   ->   4 _ 5   ->   4 5 _   ->   4 5 6
7 6 _   ->   7 _ 6   ->   7 8 6   ->   7 8 6   ->   7 8 _
costo: 4


In [8]:
#| include: false
from node import depth

amplitud = WebPuzzle(problem, "BREADTH-FIRST-SEARCH",
                     "cola FIFO, objetivo al generar",
                     key=depth, key_name="prof")
breadth_first_search(problem, view=amplitud)


<(1, 2, 3, 4, 5, 6, 7, 8, 0) g=4>

In [9]:
#| echo: false
#| column: screen-inset
ver(amplitud)

Encontró la solución de **4 movimientos** expandiendo 13 nodos.

En el árbol se ven los niveles completos, uno abajo del otro, y también los
tableros que ya habían aparecido antes: quedan de gris y con el borde punteado.
Son los que `reached` descarta. Como acá `reached` es un **conjunto de estados**
y no un diccionario de nodos, alcanza con haber visto el tablero una vez para no
volver a mirarlo.

### 2.2 — Costo uniforme (Dijkstra)

El mismo `best_first_search` de siempre, con **f(n) = g(n)**: se expande el nodo
con el camino de menor costo encontrado hasta ahora, y el objetivo se prueba **al
sacarlo** de la cola.

En Rumania esto daba un camino distinto al de amplitud, porque las rutas tenían
kilómetros distintos. Acá todos los movimientos cuestan 1, así que g(n) es la
profundidad y las dos búsquedas recorren el árbol en el mismo orden: van a
devolver el mismo camino.

In [10]:
from best_first_search import best_first_search, g

solucion_uniforme = best_first_search(problem, g)

print(lado_a_lado(*path_states(solucion_uniforme)))
print("costo:", solucion_uniforme.PATH_COST)


1 2 3   ->   1 2 3   ->   1 2 3   ->   1 2 3   ->   1 2 3
4 8 5   ->   4 8 5   ->   4 _ 5   ->   4 5 _   ->   4 5 6
7 6 _   ->   7 _ 6   ->   7 8 6   ->   7 8 6   ->   7 8 _
costo: 4


In [11]:
#| include: false
uniforme = WebPuzzle(problem, "BEST-FIRST-SEARCH", "f(n) = g(n)", key=g)
best_first_search(problem, g, view=uniforme)


<(1, 2, 3, 4, 5, 6, 7, 8, 0) g=4>

Mismo camino de 4 movimientos, pero **13 nodos contra 26**: costo uniforme
expandió el doble. ¿Por qué, si los dos recorren el árbol en el mismo orden?

In [12]:
#| echo: false
#| column: screen-inset
ver(uniforme)

La razón es la prueba de objetivo. Búsqueda en amplitud la hace **al generar** el
hijo y corta ahí; costo uniforme la hace **al sacarlo** de la cola, así que antes
de llegar a mirarlo termina de expandir todo lo que costaba menos.

En Rumania las dos daban caminos distintos —450 km contra 418— y la diferencia
quedaba tapada por esa discusión. Acá dan el mismo camino, y lo único que queda
para comparar es el trabajo.

### 2.3 — Búsqueda en profundidad

Cambiamos la FIFO por una **LIFO** y sacamos `reached`, igual que en Rumania: se
toma siempre el último hijo generado, así que la búsqueda profundiza en una rama
en vez de un nivel.

En Rumania esto daba una solución mala —575 km, 7 ciudades— pero la daba. Acá no
termina: hay que cortarla a mano para poder mirar qué estuvo haciendo.

In [13]:
from frontier import LIFOQueue
from node import Node, expand, is_cycle

# El bucle de busqueda en profundidad, cortado a mano en mil vueltas:
# sin ese corte esta celda no termina. Va sin la prueba de objetivo,
# que en mil vueltas no llega a dispararse ni una vez.
frontier = LIFOQueue([Node(STATE=problem.INITIAL)])
hondo = 0

for vuelta in range(1000):
    node = frontier.POP()
    # Cada movimiento cuesta 1, asi que g(n) es la profundidad del nodo.
    hondo = max(hondo, node.PATH_COST)
    if not is_cycle(node):
        for child in expand(problem, node):
            frontier.ADD(child)

print(f"despues de 1000 expansiones el nodo mas hondo estaba a {hondo} movimientos")
print(f"y el tablero se resuelve en {solucion_amplitud.PATH_COST}")


despues de 1000 expansiones el nodo mas hondo estaba a 633 movimientos
y el tablero se resuelve en 4


Después de mil expansiones el nodo más hondo estaba a **633 movimientos** del
inicio, y el tablero se resuelve en 4.

La pila LIFO baja por una rama y no sube nunca. `IS_CYCLE` mira solo los 30
ancestros más cercanos, y con 181.440 estados eso no alcanza: para que se
dispare, la búsqueda tiene que volver a uno de los últimos 30 tableros, y siempre
tiene otro para donde seguir.

En Rumania esto no pasaba. Con 20 ciudades, bajar 30 niveles obliga a repetir
alguna, `IS_CYCLE` corta y la búsqueda vuelve a subir. **La búsqueda en
profundidad depende de que el espacio sea chico**, y eso no está escrito en
ninguna parte del algoritmo.

Por eso el libro no la presenta sola, sino como el caso `l = ∞` de la que sigue.

### 2.4 — Profundidad limitada

La misma búsqueda en profundidad, con un techo `l`: no se expande nada más hondo
que eso. Es lo único que cambia, y es lo único que hace que termine.

```python
if depth(node) > l:       # <- el techo
    result = cutoff       # <- no encontro, pero podria haber mas abajo
elif not is_cycle(node):
    ...
```

Con `l = 5` la solución de 4 movimientos entra debajo del límite. Con `l = 3`
daría `cutoff`: la respuesta está más abajo del techo.

In [14]:
from depth_first_search import depth_limited_search

solucion_limitada = depth_limited_search(problem, 5)

print(lado_a_lado(*path_states(solucion_limitada)))
print("costo:", solucion_limitada.PATH_COST)


1 2 3   ->   1 2 3   ->   1 2 3   ->   1 2 3   ->   1 2 3
4 8 5   ->   4 8 5   ->   4 _ 5   ->   4 5 _   ->   4 5 6
7 6 _   ->   7 _ 6   ->   7 8 6   ->   7 8 6   ->   7 8 _
costo: 4


In [15]:
#| include: false
limitada = WebPuzzle(problem, "DEPTH-LIMITED-SEARCH", "pila LIFO, l = 5",
                     key=depth, key_name="prof")
depth_limited_search(problem, 5, view=limitada)


<(1, 2, 3, 4, 5, 6, 7, 8, 0) g=4>

In [16]:
#| echo: false
#| column: screen-inset
ver(limitada)

Encontró la solución de **4 movimientos**, expandiendo 51 nodos: cuatro veces
más que búsqueda en amplitud, para el mismo camino.

Baja hasta el límite, frena —el aviso rojo `pasa el limite l = 5: no se
expande`— y vuelve a subir a probar otra rama. El árbol crece para abajo y a lo
ancho al mismo tiempo, que es la diferencia con los niveles prolijos de la
búsqueda en amplitud.

### 2.5 — Profundidad iterativa

Búsqueda en profundidad limitada, pero probando `l = 0, 1, 2, ...` hasta que deja
de dar `cutoff`. Se queda con lo mejor de los dos lados: la memoria de la búsqueda
en profundidad —lineal con la profundidad— y la garantía de la búsqueda en
amplitud de encontrar primero la solución más superficial.

Acá el objetivo está a 4 movimientos, así que va a dar cinco vueltas: de `l = 0`
a `l = 4`.

In [17]:
from iterative_deepening_search import iterative_deepening_search

solucion_iterativa = iterative_deepening_search(problem)

print(lado_a_lado(*path_states(solucion_iterativa)))
print("costo:", solucion_iterativa.PATH_COST)


1 2 3   ->   1 2 3   ->   1 2 3   ->   1 2 3   ->   1 2 3
4 8 5   ->   4 8 5   ->   4 _ 5   ->   4 5 _   ->   4 5 6
7 6 _   ->   7 _ 6   ->   7 8 6   ->   7 8 6   ->   7 8 _
costo: 4


In [18]:
#| include: false
iterativa = WebPuzzle(problem, "ITERATIVE-DEEPENING-SEARCH",
                      "depth-limited con l = 0, 1, 2, ...",
                      key=depth, key_name="prof")
iterative_deepening_search(problem, view=iterativa)


<(1, 2, 3, 4, 5, 6, 7, 8, 0) g=4>

Devolvió lo mismo que la búsqueda en amplitud: 4 movimientos. Pero expandió
**51 nodos en vez de 13**. ¿Por qué?

In [19]:
#| echo: false
#| column: screen-inset
ver(iterativa)

En el riel de arriba se ven las vueltas separadas, y el árbol se rehace de cero
en cada una: eso es exactamente lo que hace el algoritmo, y ahí están los nodos
de más.

Rehacer el trabajo parece un desperdicio, pero no lo es tanto: el último nivel
tiene más nodos que todos los anteriores juntos, así que la última vuelta pesa
más que todas las otras sumadas. Lo que compra a cambio es memoria. Nunca tiene
más de `l` niveles en la pila, mientras que la búsqueda en amplitud guarda un
nivel entero.

### 2.6 — Búsqueda bidireccional

Dos búsquedas a la vez: una que sale del tablero inicial y otra que sale del
objetivo, hasta que se cruzan en algún tablero del medio.

El problema de atrás es el mismo `EightPuzzle` con `INITIAL` y `GOAL` cambiados
de lugar. En vez de un árbol de profundidad 4, dos de profundidad 2.

In [20]:
from bidirectional_search import bidirectional_search

problem_b = EightPuzzle(GOAL, CHICO)   # el mismo problema, al reves

solucion_bidireccional = bidirectional_search(problem, problem_b)

print(lado_a_lado(*path_states(solucion_bidireccional)))
print("costo:", solucion_bidireccional.PATH_COST)


1 2 3   ->   1 2 3   ->   1 2 3   ->   1 2 3   ->   1 2 3
4 8 5   ->   4 8 5   ->   4 _ 5   ->   4 5 _   ->   4 5 6
7 6 _   ->   7 _ 6   ->   7 8 6   ->   7 8 6   ->   7 8 _
costo: 4


In [21]:
#| include: false
bidireccional = WebPuzzle(problem, "BIBF-SEARCH", "dos frentes, f(n) = g(n)",
                          key=lambda node: node.PATH_COST, key_name="g")
bidirectional_search(problem, problem_b, view=bidireccional)


<(1, 2, 3, 4, 5, 6, 7, 8, 0) g=4>

In [22]:
#| echo: false
#| column: screen-inset
ver(bidireccional)

Encontró la solución de **4 movimientos** expandiendo 10 nodos: menos que
cualquiera de los otros cuatro.

Hay una condición que el 8-puzzle cumple y que la búsqueda bidireccional
necesita: **se puede dar vuelta.** Toda acción se deshace con otra acción, así
que el problema inverso es el mismo problema con `INITIAL` y `GOAL` cambiados de
lugar.

Cuando una acción no se puede deshacer, el problema inverso hay que construirlo a
mano, y a veces directamente no se puede. Esa es la limitación real de la
búsqueda bidireccional, y no aparece ni en el mapa de Rumania ni acá.

***
## Parte 3 — Comparación


In [23]:
resultados = [
    ("búsqueda en amplitud",    solucion_amplitud,      amplitud),
    ("costo uniforme",          solucion_uniforme,      uniforme),
    ("profundidad limitada",    solucion_limitada,      limitada),
    ("profundidad iterativa",   solucion_iterativa,     iterativa),
    ("búsqueda bidireccional",  solucion_bidireccional, bidireccional),
]

# El costo del camino es la cantidad de movimientos: cada uno cuesta 1.
print(f"{'algoritmo':<25}{'movimientos':>13}{'expande':>9}")
print("-" * 47)
for nombre, solucion, view in resultados:
    print(f"{nombre:<25}{len(path_states(solucion)) - 1:>13}{view.pops:>9}")
print(f"{'búsqueda en profundidad':<25}{'no termina':>13}{'-':>9}")

algoritmo                  movimientos  expande
-----------------------------------------------
búsqueda en amplitud                 4       13
costo uniforme                       4       26
profundidad limitada                 4       51
profundidad iterativa                4       51
búsqueda bidireccional               4       10
búsqueda en profundidad     no termina        -


¿Qué observaciones se pueden hacer de estos resultados?


::: {.content-hidden}
Observaciones:

- los cinco que terminan dan **la misma respuesta**, cuatro movimientos: con
  costos uniformes cualquiera de ellos encuentra el óptimo;
- lo que los separa es el trabajo, entre **10 y 51 expansiones**;
- **búsqueda en profundidad** no da respuesta.
:::


Con un tablero a cuatro movimientos del objetivo esas diferencias son
chicas. Repitamos la tabla sobre el tablero del libro, que está a veintiuno.


In [24]:
from puzzle import FIGURA_34, imprimir

imprimir(FIGURA_34, tope=200_000,
         titulo="Figura 3.4 del libro: a veintiun movimientos")

Figura 3.4 del libro: a veintiun movimientos

7 2 4   ->   1 2 3
_ 5 6   ->   4 5 6
8 3 1   ->   7 8 _

algoritmo                  pasos  expande  más hondo
----------------------------------------------------


búsqueda en amplitud          21    47514         20
costo uniforme                21    64732         21
búsqueda en profundidad        -     4690       2967
profundidad iterativa          -   200000         18
búsqueda bidireccional        21     1477         10
  - búsqueda en profundidad: se quedó sin pila a 2967 niveles
  - profundidad iterativa: llegó al tope de 200000 expansiones



Veintiún movimientos en vez de cuatro, y la tabla se da vuelta:

- **búsqueda bidireccional** pasa a ser la única razonable: **1.477** nodos
  contra 47.514. Dos árboles de profundidad 10 en vez de uno de profundidad 21,
  que es la diferencia entre O(b^(d/2)) y O(b^d). En Rumania ganaba por 11 a 13 y
  parecía una curiosidad;
- **profundidad iterativa** tampoco termina: llegó al tope de 200.000
  expansiones. Su ventaja es la memoria y la paga en tiempo, y con un árbol así
  de grande ese precio se vuelve impagable;
- **costo uniforme** sigue expandiendo más que búsqueda en amplitud —64.732
  contra 47.514— por lo mismo de siempre: la prueba de objetivo tardía;
- **búsqueda en profundidad** se quedó sin pila a casi 3.000 niveles de hondo, buscando
  un tablero que está a 21.

Con 20 ciudades todos los algoritmos terminan y las diferencias son de un nodo o
dos. El 8-puzzle tiene 181.440 estados —que sigue siendo un problema de
juguete— y ya alcanza para que dos de los cinco se queden sin dar respuesta.

***
## Rumania y el 8-puzzle

| | Rumania | 8-puzzle |
|---|---|---|
| el grafo | escrito, 23 rutas | no existe: lo calcula `ACTIONS` |
| tamaño | 20 estados | 181.440 estados |
| costos | distintos (75 a 211 km) | todos 1 |

Consecuencias:

1. **El grafo no hace falta.** Un algoritmo de búsqueda nunca pide el grafo
   entero, pide los vecinos de un estado por vez. Por eso sirve en espacios que
   no entrarían en memoria.
2. **La búsqueda en profundidad no es un algoritmo completo.** Sin límite y sin
   `reached`, depende de que el espacio sea chico. Profundidad limitada y
   profundidad iterativa no son variantes: son lo que la hace usable.
3. **Con costos uniformes, amplitud y costo uniforme dan lo mismo.** Toda la
   diferencia entre las dos está en cuándo se prueba el objetivo.